# 02 – Training & Evaluation

Walkthrough of the full training loop using the Trainer class,
followed by evaluation on the test set and visualisation of results.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import yaml
import torch

from models.factory import build_model, get_device
from data.dataset import create_dataloaders
from training.trainer import Trainer
from training.visualization import plot_training_curves

with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

# Quick-train override for notebook demo
cfg['training']['epochs'] = 5
cfg['data']['batch_size'] = 4

device = get_device('auto')
print(f'Using device: {device}')

In [ ]:
cfg['model']['num_classes'] = cfg['data']['num_classes']
cfg['model']['in_channels'] = cfg['data']['num_channels']

model = build_model(cfg['model']).to(device)
print(model)
print(f'Parameters: {model.count_parameters():,}')

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(cfg)
print(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}')

In [ ]:
trainer = Trainer(model=model, train_loader=train_loader,
                  val_loader=val_loader, cfg=cfg, device=device)
history = trainer.train()

In [ ]:
plot_training_curves(history, save_path=None)

In [ ]:
from training.metrics import SegmentationMetrics

seg_metrics = SegmentationMetrics(threshold=0.5)
model.eval()

with torch.no_grad():
    for batch in test_loader:
        images  = batch['image'].to(device)
        targets = batch['mask'].to(device)
        logits  = model(images)
        seg_metrics.update(logits, targets)

results = seg_metrics.compute()
print('\n=== Test Metrics ===')
for k, v in results.items():
    if v is not None:
        print(f'  {k:<20}: {v:.4f}')